# RADS Verification Notebook 04: Training Sanity Check

This notebook runs a full 1-epoch training loop to verify that the `Trainer` and `Evaluator` work end-to-end, saves model checkpoints, resumes training from a checkpoint, saves predictions, and logs all metrics to W&B.


In [6]:
import os
import sys
import shutil
from pathlib import Path
import torch

# Ensure project root is in path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from training.configs.config import load_training_config
from training.utils.seed import set_global_seed
from training.utils.device import get_device
from training.utils.logger import TrainingLogger
from training.utils.output_manager import TrainingOutputManager
from training.utils.wandb_manager import TrainingWandbManager
from training.datasets.dataloader import create_dataloaders
from training.models.model_factory import create_model
from training.losses.classification_loss import create_loss
from training.callbacks.checkpoint import CheckpointManager
from training.engine.trainer import Trainer
from training.engine.evaluator import Evaluator

# Load configuration and override epochs to 1 for sanity check
config = load_training_config()
print('Sanity Check configurations loaded.')


Sanity Check configurations loaded.


In [7]:
# Initialization
set_global_seed(config.seed)
device = get_device()

# Setup Logger and Output Manager
tlogger = TrainingLogger.setup(config)
output_mgr = TrainingOutputManager(config)


In [8]:
# Load DataLoaders
train_loader, val_loader = create_dataloaders(config, use_weighted_sampler=True)
class_names = train_loader.dataset.class_names


In [9]:
# Create Model, Loss, Optimizer, and Scheduler
model = create_model(config)
class_weights = train_loader.dataset.class_weights
loss_fn = create_loss(config, class_weights=class_weights, device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Checkpoint manager
ckpt_mgr = CheckpointManager(
    checkpoint_dir=output_mgr.checkpoints_dir,
    monitor_metric=config.monitor_metric,
    mode=config.monitor_mode,
    save_best=config.save_best,
    save_last=config.save_last,
)


In [10]:
# Initialize W&B and run Trainer
wb_manager = TrainingWandbManager(config)

with wb_manager:
    # We pass the callbacks directly to the trainer
    trainer = Trainer(
        model=model,
        loss_fn=loss_fn,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        training_logger=tlogger,
        wandb_manager=wb_manager,
        checkpoint_manager=ckpt_mgr,
        class_names=class_names,
    )
    
    print('Starting 1-epoch sanity check training...')
    history = trainer.fit(start_epoch=0)
    print('Training finished!')


wandb: Currently logged in as: saksham4data (saksham4data-vinkura) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting 1-epoch sanity check training...
[01:27:56] INFO     === TRAINING START: sanity_check_run === | v1.0.0 | Python 3.12.4 | Windows
[01:27:56] INFO     -- Epoch 1/1 ------------------------------------------------


Train Epoch 1:   0%|          | 0/13 [00:00<?, ?it/s]Failed to load frame 85 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v19.mov: Frame extraction failed after both seeking and sequential decoding attempts. Skipping sample.
Failed to load frame 67 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v19.mov: Frame extraction failed after both seeking and sequential decoding attempts. Skipping sample.
Failed to load frame 106 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v44.mov: Frame extraction failed after both seeking and sequential decoding attempts. Skipping sample.
Train Epoch 1:  31%|███       | 4/13 [01:50<03:59, 26.56s/it, loss=1.0733]Failed to load frame 106 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v44.mov: Frame extraction failed after both seeking and sequential decoding attempts. Skipping sample.
Failed to load frame 85 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v19.mov: Frame extraction failed a

[01:34:30] INFO     Epoch 1 complete | train_loss=1.0181 | val_loss=2.0519 | lr=0.001000 | time=392.6s
Training finished!


epoch,▁▁
lr/learning_rate,▁
train/f1_accident,▁
train/f1_non_accident,▁
train/loss,▁
train/macro_f1,▁
train/precision_accident,▁
train/precision_non_accident,▁
train/recall_accident,▁
train/recall_non_accident,▁
+10,...


In [11]:
# Verify checkpoints were created
best_ckpt_path = output_mgr.checkpoints_dir / 'best.pt'
last_ckpt_path = output_mgr.checkpoints_dir / 'last.pt'

assert best_ckpt_path.is_file(), 'best.pt not found!'
assert last_ckpt_path.is_file(), 'last.pt not found!'
print('Checkpoints saved successfully!')


Checkpoints saved successfully!


In [12]:
# Verification: Resume from checkpoint
print('Verifying checkpoint resumption...')
resumed_model = create_model(config)
resumed_opt = torch.optim.Adam(resumed_model.parameters(), lr=config.learning_rate)
resumed_scheduler = torch.optim.lr_scheduler.StepLR(resumed_opt, step_size=5, gamma=0.1)

resumed_epoch = ckpt_mgr.resume_from(
    best_ckpt_path,
    resumed_model,
    resumed_opt,
    resumed_scheduler
)
assert resumed_epoch == 1, f'Expected resumed epoch 1, got {resumed_epoch}'
print('Resumption verified successfully!')


Verifying checkpoint resumption...
Resumption verified successfully!


In [13]:
# Verification: Save Predictions using Evaluator
evaluator = Evaluator(resumed_model, loss_fn, device, class_names)
metrics = evaluator.evaluate(
    val_loader,
    split_name='val',
    output_manager=output_mgr,
    wandb_manager=None
)

# Check predictions outputs
val_preds_path = output_mgr.predictions_dir / 'val_predictions.json'
val_cm_path = output_mgr.predictions_dir / 'val_confusion_matrix.json'

assert val_preds_path.is_file(), 'val_predictions.json not found!'
assert val_cm_path.is_file(), 'val_confusion_matrix.json not found!'
print('Prediction outputs verified successfully!')


Evaluating (val):  33%|███▎      | 1/3 [00:19<00:39, 19.75s/it, loss=4.3602]Failed to load frame 174 from E:\Rads\Datasets\raw\tudat\Final_videos\Negative_Videos\v46.mov: Frame extraction failed after both seeking and sequential decoding attempts. Skipping sample.
                                                                            

Prediction outputs verified successfully!


In [14]:
# Finalize output directory and write run manifest
manifest = output_mgr.finalize(
    wandb_run_id=None,
    wandb_run_url=None,
    git_commit=TrainingLogger.get_git_commit(),
    total_epochs=1,
    best_metric=ckpt_mgr.best_value,
    best_epoch=ckpt_mgr.best_epoch,
    duration_seconds=10.0,
    status='success'
)
print('Sanity Check run finalized.')
print(f'Manifest contents:\n{manifest}')


Sanity Check run finalized.
Manifest contents:
{'timestamp': '2026-08-07T19:57:47.039715+00:00', 'run_name': 'sanity_check_run', 'training_version': '1.0.0', 'model': 'resnet18', 'dataset': 'tudat', 'git_commit': '1c0a1acd44bf4c21aae84ac086431dd607c5ea65', 'seed': 42, 'wandb_run_id': None, 'wandb_run_url': None, 'files': {'checkpoints': [], 'metrics': [], 'predictions': ['val_predictions.json', 'val_confusion_matrix.json'], 'logs': []}, 'execution': {'total_epochs': 1, 'best_metric': 2.0519490838050842, 'best_epoch': 0, 'duration_seconds': 10.0, 'status': 'success'}}
